# 11 — 1D CNN Proof-of-Concept

Tests whether a 1D Convolutional Neural Network can improve over the classical
classifiers (XGBoost, SVM, RF, LR) on the **polynomial coefficient representations**
(Chebyshev & Legendre, L2-normalised).

**Motivation:** The original article (Ambrosch et al. 2026) achieved its best results
with a CNN (ROC AUC 0.961) on XP coefficients. Polynomial coefficients have a
natural ordering by degree — low-degree coefficients capture coarse spectral shape,
higher degrees capture finer detail — making them a reasonable input for 1D
convolutions that learn local multi-scale patterns.

**Scope:** POC only — uses rep0 (5 folds) for speed. If results are promising,
the CNN will be added to the full `05_classify_focused.py` experiment.

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")

device = torch.device("cpu")
print(f"PyTorch {torch.__version__}, device: {device}")

PyTorch 2.11.0, device: cpu


## 1. Load splits and define representations

In [2]:
with open(DATA_DIR / "splits_rskf.json") as f:
    all_splits = json.load(f)

splits = {k: v for k, v in all_splits.items() if k.startswith("rep0_")}
print(f"Using {len(splits)} folds (rep0 only)")

REPRESENTATIONS = []
for basis in ["chebyshev", "legendre"]:
    for n in [5, 10, 20, 50]:
        REPRESENTATIONS.append({
            "name": f"{basis}_{n}_L2",
            "file": f"{basis}_{n}_L2.csv",
            "n_features": n,
        })

print(f"\n{len(REPRESENTATIONS)} representations:")
for r in REPRESENTATIONS:
    print(f"  {r['name']:25s} ({r['n_features']:3d} features)")

Using 5 folds (rep0 only)

8 representations:
  chebyshev_5_L2            (  5 features)
  chebyshev_10_L2           ( 10 features)
  chebyshev_20_L2           ( 20 features)
  chebyshev_50_L2           ( 50 features)
  legendre_5_L2             (  5 features)
  legendre_10_L2            ( 10 features)
  legendre_20_L2            ( 20 features)
  legendre_50_L2            ( 50 features)


## 2. 1D CNN model

In [3]:
class SpectralCNN(nn.Module):
    """Small 1D CNN for coefficient-based binary classification."""

    def __init__(self, n_features: int):
        super().__init__()
        k1 = min(5, n_features)
        k2 = min(3, n_features)
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=k1, padding=k1 // 2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=k2, padding=k2 // 2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.squeeze(-1)
        return self.head(x).squeeze(-1)


n_params = sum(p.numel() for p in SpectralCNN(20).parameters())
print(f"Model parameters (n_features=20): {n_params:,}")

Model parameters (n_features=20): 8,705


## 3. Training and evaluation helpers

In [4]:
def pick_youden_threshold(y_true, y_prob, grid_size=200):
    thresholds = np.linspace(0, 1, grid_size)
    best_j, best_thr = -1, 0.5
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0
        spec = tn / (tn + fp) if (tn + fp) else 0
        j = sens + spec - 1
        if j > best_j:
            best_j, best_thr = j, thr
    return best_thr


def evaluate(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn)
    f1 = (2 * prec * sens) / (prec + sens) if (prec + sens) else 0.0
    return {
        "threshold": threshold,
        "sensitivity": sens,
        "specificity": spec,
        "precision": prec,
        "accuracy": acc,
        "f1": f1,
        "youden_j": sens + spec - 1.0,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "log_loss": log_loss(y_true, y_prob),
    }

In [5]:
def train_one_fold(
    X_all, y_all, train_idx, test_idx, n_features,
    lr=1e-3, weight_decay=1e-4, max_epochs=200, patience=20, batch_size=64,
    val_frac=0.15,
):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_all[train_idx])
    X_te = scaler.transform(X_all[test_idx])
    y_tr = y_all[train_idx]
    y_te = y_all[test_idx]

    # Validation split (stratified)
    n_val = int(len(X_tr) * val_frac)
    rng = np.random.RandomState(RANDOM_STATE)
    idx = rng.permutation(len(X_tr))
    pos = idx[y_tr[idx] == 1]
    neg = idx[y_tr[idx] == 0]
    n_val_pos = max(1, int(len(pos) * val_frac))
    n_val_neg = n_val - n_val_pos
    val_idx = np.concatenate([pos[:n_val_pos], neg[:n_val_neg]])
    tr_idx = np.concatenate([pos[n_val_pos:], neg[n_val_neg:]])

    X_val, y_val = X_tr[val_idx], y_tr[val_idx]
    X_tr_final, y_tr_final = X_tr[tr_idx], y_tr[tr_idx]

    # Tensors: reshape to (N, 1, D) for Conv1d
    def to_tensor(X, y=None):
        t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        if y is not None:
            return t, torch.tensor(y, dtype=torch.float32)
        return t

    X_tr_t, y_tr_t = to_tensor(X_tr_final, y_tr_final)
    X_val_t, y_val_t = to_tensor(X_val, y_val)
    X_te_t, y_te_t = to_tensor(X_te, y_te)

    # Class weight for imbalanced data (~80/20)
    n_pos = y_tr_final.sum()
    n_neg = len(y_tr_final) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

    model = SpectralCNN(n_features).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(len(X_tr_t))
        epoch_loss = 0.0
        n_batches = 0
        for i in range(0, len(X_tr_t), batch_size):
            batch_idx = perm[i : i + batch_size]
            xb, yb = X_tr_t[batch_idx], y_tr_t[batch_idx]
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        # Validation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    # Load best model and predict
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        # OOF predictions on validation set for threshold tuning
        val_probs = torch.sigmoid(model(X_val_t)).numpy()
        thr = pick_youden_threshold(y_val, val_probs)

        # Test predictions
        test_probs = torch.sigmoid(model(X_te_t)).numpy()

    metrics = evaluate(y_te, test_probs, thr)
    metrics["stopped_epoch"] = epoch + 1
    metrics["best_val_loss"] = best_val_loss
    return metrics

## 4. Run POC experiment

In [6]:
all_results = []

for repr_cfg in REPRESENTATIONS:
    repr_file = DATA_DIR / repr_cfg["file"]
    if not repr_file.exists():
        print(f"SKIP — {repr_file.name} not found")
        continue

    df = pd.read_csv(repr_file)
    feat_cols = [c for c in df.columns if c not in ("source_id", "y")]
    X_all = df[feat_cols].to_numpy(dtype=np.float64)
    y_all = df["y"].to_numpy(dtype=int)

    print(f"\n{'─'*60}")
    print(f"  {repr_cfg['name']}  ({repr_cfg['n_features']} features)")
    print(f"{'─'*60}")

    for split_name, split_idx in splits.items():
        torch.manual_seed(RANDOM_STATE)
        np.random.seed(RANDOM_STATE)

        train_idx = np.array(split_idx["train"])
        test_idx = np.array(split_idx["test"])

        metrics = train_one_fold(
            X_all, y_all, train_idx, test_idx,
            n_features=repr_cfg["n_features"],
        )
        metrics["representation"] = repr_cfg["name"]
        metrics["n_features"] = repr_cfg["n_features"]
        metrics["split"] = split_name
        all_results.append(metrics)

        print(
            f"  {split_name}  ROC-AUC={metrics['roc_auc']:.4f}  "
            f"Sens={metrics['sensitivity']:.4f}  "
            f"Prec={metrics['precision']:.4f}  "
            f"ep={metrics['stopped_epoch']}"
        )

df_cnn = pd.DataFrame(all_results)
print(f"\nCollected {len(df_cnn)} results.")


────────────────────────────────────────────────────────────
  chebyshev_5_L2  (5 features)
────────────────────────────────────────────────────────────
  rep0_fold0  ROC-AUC=0.9413  Sens=0.8829  Prec=0.8305  ep=71
  rep0_fold1  ROC-AUC=0.8963  Sens=0.7297  Prec=0.8182  ep=49
  rep0_fold2  ROC-AUC=0.9214  Sens=0.8125  Prec=0.8273  ep=62
  rep0_fold3  ROC-AUC=0.9302  Sens=0.8839  Prec=0.7226  ep=57
  rep0_fold4  ROC-AUC=0.8875  Sens=0.8304  Prec=0.7686  ep=52

────────────────────────────────────────────────────────────
  chebyshev_10_L2  (10 features)
────────────────────────────────────────────────────────────
  rep0_fold0  ROC-AUC=0.9447  Sens=0.8919  Prec=0.7071  ep=57
  rep0_fold1  ROC-AUC=0.8933  Sens=0.7297  Prec=0.7788  ep=83
  rep0_fold2  ROC-AUC=0.9149  Sens=0.8304  Prec=0.7623  ep=72
  rep0_fold3  ROC-AUC=0.8969  Sens=0.8125  Prec=0.7583  ep=128
  rep0_fold4  ROC-AUC=0.9011  Sens=0.8036  Prec=0.7143  ep=77

────────────────────────────────────────────────────────────
  cheby

## 5. CNN summary (mean across 5 folds)

In [7]:
metric_cols = ["roc_auc", "pr_auc", "sensitivity", "specificity", "precision", "f1", "youden_j"]

cnn_summary = (
    df_cnn
    .groupby(["representation", "n_features"])[metric_cols]
    .agg(["mean", "std"])
)
cnn_summary.columns = [f"{col}_{stat}" for col, stat in cnn_summary.columns]
cnn_summary = cnn_summary.reset_index().sort_values("roc_auc_mean", ascending=False)

display_cols = ["representation", "n_features", "roc_auc_mean", "roc_auc_std",
                "sensitivity_mean", "precision_mean", "f1_mean"]
cnn_summary[display_cols].round(4)

,representation,n_features,roc_auc_mean,roc_auc_std,sensitivity_mean,precision_mean,f1_mean
7,legendre_5_L2,5,0.9169,0.0240,0.8279,0.7977,0.8093
3,chebyshev_5_L2,5,0.9153,0.0228,0.8279,0.7934,0.8081
0,chebyshev_10_L2,10,0.9102,0.0210,0.8136,0.7442,0.7756
4,legendre_10_L2,10,0.9102,0.0236,0.8208,0.7509,0.7829
1,chebyshev_20_L2,20,0.9078,0.0220,0.7868,0.6515,0.7106
5,legendre_20_L2,20,0.9023,0.0251,0.7813,0.7629,0.7687
2,chebyshev_50_L2,50,0.8973,0.0202,0.7313,0.7214,0.7234
6,legendre_50_L2,50,0.8914,0.0250,0.7545,0.6916,0.7202


## 6. Comparison with classical models

In [8]:
focused = pd.read_csv(RESULTS_DIR / "focused_summary.csv")

poc_reprs = df_cnn["representation"].unique()
classical = focused[focused["representation"].isin(poc_reprs)].copy()

# Best classical model per representation
best_classical = (
    classical
    .sort_values("roc_auc_mean", ascending=False)
    .groupby("representation")
    .first()
    .reset_index()
)
best_classical["model"] = best_classical["classifier"]

# CNN summary with label
cnn_comp = cnn_summary.copy()
cnn_comp["model"] = "CNN_1D"

compare_cols = ["representation", "n_features", "model",
                "roc_auc_mean", "roc_auc_std",
                "sensitivity_mean", "precision_mean", "f1_mean"]

comparison = pd.concat([
    best_classical[compare_cols],
    cnn_comp[compare_cols],
]).sort_values(["representation", "model"])

print("CNN vs Best Classical Model per Representation")
print("=" * 70)
comparison.round(4)

CNN vs Best Classical Model per Representation


,representation,n_features,model,roc_auc_mean,roc_auc_std,sensitivity_mean,precision_mean,f1_mean
0,chebyshev_10_L2,10,CNN_1D,0.9102,0.0210,0.8136,0.7442,0.7756
0,chebyshev_10_L2,10,SVM_RBF,0.9241,0.0020,0.8332,0.7638,0.7960
1,chebyshev_20_L2,20,CNN_1D,0.9078,0.0220,0.7868,0.6515,0.7106
1,chebyshev_20_L2,20,LogisticRegression,0.9164,0.0021,0.8328,0.7828,0.8063
2,chebyshev_50_L2,50,CNN_1D,0.8973,0.0202,0.7313,0.7214,0.7234
2,chebyshev_50_L2,50,LogisticRegression,0.9174,0.0031,0.8337,0.7786,0.8043
3,chebyshev_5_L2,5,CNN_1D,0.9153,0.0228,0.8279,0.7934,0.8081
3,chebyshev_5_L2,5,SVM_RBF,0.9274,0.0016,0.8335,0.7915,0.8112
4,legendre_10_L2,10,CNN_1D,0.9102,0.0236,0.8208,0.7509,0.7829
5,legendre_20_L2,20,CNN_1D,0.9023,0.0251,0.7813,0.7629,0.7687


In [10]:
# Delta table: CNN minus best classical
delta_rows = []
for repr_name in poc_reprs:
    cnn_match = cnn_comp[cnn_comp["representation"] == repr_name]
    cls_match = best_classical[best_classical["representation"] == repr_name]
    if cnn_match.empty or cls_match.empty:
        continue
    cnn_row = cnn_match.iloc[0]
    cls_row = cls_match.iloc[0]
    delta_rows.append({
        "representation": repr_name,
        "best_classical": cls_row["model"],
        "delta_roc_auc": cnn_row["roc_auc_mean"] - cls_row["roc_auc_mean"],
        "delta_sensitivity": cnn_row["sensitivity_mean"] - cls_row["sensitivity_mean"],
        "delta_precision": cnn_row["precision_mean"] - cls_row["precision_mean"],
        "delta_f1": cnn_row["f1_mean"] - cls_row["f1_mean"],
    })

df_delta = pd.DataFrame(delta_rows)
print("Delta: CNN − Best Classical (positive = CNN wins)")
print(f"({len(delta_rows)}/{len(poc_reprs)} representations matched)")
print("=" * 60)
df_delta.round(4)

Delta: CNN − Best Classical (positive = CNN wins)
(5/8 representations matched)


,representation,best_classical,delta_roc_auc,delta_sensitivity,delta_precision,delta_f1
0,chebyshev_5_L2,SVM_RBF,-0.0120,-0.0056,0.0019,-0.0031
1,chebyshev_10_L2,SVM_RBF,-0.0139,-0.0195,-0.0197,-0.0204
2,chebyshev_20_L2,LogisticRegression,-0.0085,-0.0460,-0.1313,-0.0957
3,chebyshev_50_L2,LogisticRegression,-0.0201,-0.1024,-0.0571,-0.0809
4,legendre_50_L2,LogisticRegression,-0.0267,-0.0731,-0.0918,-0.0835


## 7. Verdict

In [11]:
avg_delta = df_delta["delta_roc_auc"].mean()
wins = (df_delta["delta_roc_auc"] > 0).sum()
total = len(df_delta)

print(f"Average ROC AUC delta (CNN - classical): {avg_delta:+.4f}")
print(f"CNN wins on {wins}/{total} representations")
print()
if wins > total / 2 and avg_delta > 0:
    print("VERDICT: CNN shows promise. Worth adding to the full experiment.")
elif wins > 0:
    print("VERDICT: Mixed results. CNN helps on some representations but not all.")
    print("Consider running full 10x5 CV before deciding.")
else:
    print("VERDICT: CNN does not improve over classical models on polynomial features.")
    print("The sequential structure may not provide enough benefit at these dimensions.")

Average ROC AUC delta (CNN - classical): -0.0163
CNN wins on 0/5 representations

VERDICT: CNN does not improve over classical models on polynomial features.
The sequential structure may not provide enough benefit at these dimensions.
